In [20]:
import json
import pandas as pd
from pprint import pprint
import matplotlib.pyplot as plt
import networkx as nx
import seaborn as sns
import plotly.express as px

Load the JSON and create a master DataFrame of STIX objects

In [21]:
enterprise_file = "../../enterprise-attack/enterprise-attack.json"
mobile_file = "../../mobile-attack/mobile-attack.json"
ics_file = "../../ics-attack/ics-attack.json"

with open(enterprise_file, "r", encoding="utf-8") as f:
    stix = json.load(f)

objects = stix.get("objects", [])
# Master dataframe (keeps the raw dict for reference)
master_df = pd.DataFrame([{
    "id": o.get("id"),
    "type": o.get("type"),
    "name": o.get("name"),
    "description": o.get("description"),
    "created": o.get("created"),
    "modified": o.get("modified"),
    "raw": o
} for o in objects])

master_df.head()


,id,type,name,description,created,modified,raw
0,x-mitre-collection--1f5f1533-f617-4ca8-9ab4-6a...,x-mitre-collection,Enterprise ATT&CK,ATT&CK for Enterprise provides a knowledge bas...,2018-01-17T12:56:55.080Z,2025-10-28T14:00:00.188Z,"{'type': 'x-mitre-collection', 'id': 'x-mitre-..."
1,x-mitre-matrix--eafc1b4c-5e56-4965-bd4e-66a6a8...,x-mitre-matrix,Enterprise ATT&CK,Below are the tactics and technique representi...,2018-10-17T00:14:20.652Z,2025-04-25T14:41:40.982Z,"{'type': 'x-mitre-matrix', 'spec_version': '2...."
2,course-of-action--00d7d21b-69d6-4797-88a2-c86f...,course-of-action,Password Filter DLL Mitigation,Ensure only valid password filters are registe...,2018-10-17T00:14:20.652Z,2025-04-18T17:59:39.912Z,"{'type': 'course-of-action', 'spec_version': '..."
3,course-of-action--02f0f92a-0a51-4c94-9bda-6437...,course-of-action,Space after Filename Mitigation,Prevent files from having a trailing space aft...,2018-10-17T00:14:20.652Z,2025-04-18T17:59:40.127Z,"{'type': 'course-of-action', 'spec_version': '..."
4,course-of-action--03c0c586-50ed-45a7-95f4-f496...,course-of-action,HISTCONTROL Mitigation,Prevent users from changing the <code>HISTCONT...,2018-10-17T00:14:20.652Z,2025-04-18T17:59:40.291Z,"{'type': 'course-of-action', 'spec_version': '..."


Filter for Intrusion Sets


In [22]:
# Filter only intrusion-set objects
apts_df = master_df[master_df["type"] == "intrusion-set"].copy()

print(f"Found {len(apts_df)} intrusion sets (APT groups).")
apts_df.head()

# Extract readable columns for APT analysis
apts_df["aliases"] = apts_df["raw"].apply(
    lambda o: ", ".join(o.get("aliases", [])) if isinstance(o, dict) else ""
)
apts_df["url"] = apts_df["raw"].apply(
    lambda o: next(
        (ref.get("url") for ref in o.get("external_references", []) if "url" in ref),
        None
    )
)

apt_text_df = apts_df[["id", "name", "aliases", "description", "url"]].reset_index(drop=True)
apt_text_df.head(10)

Found 187 intrusion sets (APT groups).


,id,name,aliases,description,url
0,intrusion-set--01e28736-2ffc-455b-9880-ed4d140...,Indrik Spider,"Indrik Spider, Evil Corp, Manatee Tempest, DEV...",[Indrik Spider](https://attack.mitre.org/group...,https://attack.mitre.org/groups/G0119
1,intrusion-set--b7f627e2-0817-4cd5-8d50-e75f8aa...,LuminousMoth,LuminousMoth,[LuminousMoth](https://attack.mitre.org/groups...,https://attack.mitre.org/groups/G1014
2,intrusion-set--918da025-04bd-48af-b6c4-f3e4d1b...,Medusa Group,Medusa Group,[Medusa Group](https://attack.mitre.org/groups...,https://attack.mitre.org/groups/G1051
3,intrusion-set--dd2d9ca6-505b-4860-a604-233685b...,Wizard Spider,"Wizard Spider, UNC1878, TEMP.MixMaster, Grim S...",[Wizard Spider](https://attack.mitre.org/group...,https://attack.mitre.org/groups/G0102
4,intrusion-set--03506554-5f37-4f8f-9ce4-0e9f01a...,Elderwood,"Elderwood, Elderwood Gang, Beijing Group, Snea...",[Elderwood](https://attack.mitre.org/groups/G0...,https://attack.mitre.org/groups/G0066
5,intrusion-set--6b1b551c-d770-4f95-8cfc-3cd253c...,Frankenstein,Frankenstein,[Frankenstein](https://attack.mitre.org/groups...,https://attack.mitre.org/groups/G0101
6,intrusion-set--3753cc21-2dae-4dfb-8481-d004e74...,FIN7,"FIN7, GOLD NIAGARA, ITG14, Carbon Spider, ELBR...",[FIN7](https://attack.mitre.org/groups/G0046) ...,https://attack.mitre.org/groups/G0046
7,intrusion-set--461b8e25-8f4a-4ea2-a4a8-e39df7c...,UNC3886,UNC3886,[UNC3886](https://attack.mitre.org/groups/G104...,https://attack.mitre.org/groups/G1048
8,intrusion-set--e1fc262c-dad2-4b82-abda-5f08dd1...,Velvet Ant,Velvet Ant,[Velvet Ant](https://attack.mitre.org/groups/G...,https://attack.mitre.org/groups/G1047
9,intrusion-set--f8cb7b36-62ef-4488-8a6d-a7033e3...,WIRTE,WIRTE,[WIRTE](https://attack.mitre.org/groups/G0090)...,https://attack.mitre.org/groups/G0090


Export the data

In [23]:
apt_text_df.to_csv("apt_descriptions.csv", index=False)

Load in new data

In [24]:
apt_file = "apt_descriptions_tagged.csv"

apt_df = pd.read_csv(apt_file)

# String clean up
apt_df["country"] = apt_df["country"].str.replace(r"[\[\]']", "", regex=True)
apt_df["region"]  = apt_df["region"].str.replace(r"[\[\]']", "", regex=True)

apt_df.head()

,id,name,aliases,description,url,country,region,rationale
0,intrusion-set--01e28736-2ffc-455b-9880-ed4d140...,Indrik Spider,"Indrik Spider, Evil Corp, Manatee Tempest, DEV...",[Indrik Spider](https://attack.mitre.org/group...,https://attack.mitre.org/groups/G0119,Russia,Eastern Europe,"Aliases/identifiers (e.g., 'evil corp') common..."
1,intrusion-set--b7f627e2-0817-4cd5-8d50-e75f8aa...,LuminousMoth,LuminousMoth,[LuminousMoth](https://attack.mitre.org/groups...,https://attack.mitre.org/groups/G1014,China,East Asia,"Aliases/identifiers (e.g., 'mustang panda') co..."
2,intrusion-set--918da025-04bd-48af-b6c4-f3e4d1b...,Medusa Group,Medusa Group,[Medusa Group](https://attack.mitre.org/groups...,https://attack.mitre.org/groups/G1051,Unknown,Unknown,No clear country attribution in the descriptio...
3,intrusion-set--dd2d9ca6-505b-4860-a604-233685b...,Wizard Spider,"Wizard Spider, UNC1878, TEMP.MixMaster, Grim S...",[Wizard Spider](https://attack.mitre.org/group...,https://attack.mitre.org/groups/G0102,Russia,Eastern Europe,"Aliases/identifiers (e.g., 'wizard spider') co..."
4,intrusion-set--03506554-5f37-4f8f-9ce4-0e9f01a...,Elderwood,"Elderwood, Elderwood Gang, Beijing Group, Snea...",[Elderwood](https://attack.mitre.org/groups/G0...,https://attack.mitre.org/groups/G0066,China,East Asia,"Aliases/identifiers (e.g., 'elderwood') common..."


Count APTs per country

In [25]:
# Split on commas, strip spaces, and explode
apt_df["country_list"] = apt_df["country"].str.split(",")
apt_df = apt_df.explode("country_list")
apt_df["country_list"] = apt_df["country_list"].str.strip()

# Optional: remove blanks
apt_df = apt_df[apt_df["country_list"].notna() & (apt_df["country_list"] != "")]

country_counts = apt_df["country_list"].value_counts().reset_index()
country_counts.columns = ["country", "apt_count"]
country_counts.head()

,country,apt_count
0,China,72
1,Unknown,54
2,Russia,37
3,Iran,22
4,United States,16


## Analyses and Plots

Global choropleth map

In [36]:
fig = px.choropleth(
    country_counts,
    locations="country",
    locationmode="country names",
    color="apt_count",
    hover_name="country",
    color_continuous_scale="Reds",
    title="Number of APT Groups by Country (MITRE ATT&CK)",
    #width=1000,    # pixels
    #height=550     # pixels
)

# Graph modernization
fig.update_geos(
    projection_type="natural earth",  # alternatives: "mercator", "orthographic", "equirectangular"
    showcountries=True,
    showcoastlines=True,
    coastlinecolor="LightGray",
    showframe=False,
)

# Adjust margins, background, and colorbar
fig.update_layout(
    geo=dict(bgcolor="rgba(0,0,0,0)"),  # transparent map background
    margin={"r":0,"t":40,"l":0,"b":0},
    coloraxis_colorbar=dict(
        title="APT Count",
        ticksuffix=" groups"
    )
)

# Add custom title formatting
fig.update_layout(
    title=dict(
        text="Number of APT Groups by Country (MITRE ATT&CK)",
        font=dict(size=22, family="Arial", color="black"),
        x=0.5,  # center the title
        xanchor="center"
    )
)

fig.update_layout(geo=dict(showframe=False, showcoastlines=True))
fig.show()

/tmp/ipykernel_11972/1431077785.py:1: DeprecationWarning:

The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.



Bar Graph

In [38]:
fig = px.bar(
    country_counts.sort_values("apt_count", ascending=False),
    x="country",
    y="apt_count",
    text="apt_count",
    color="apt_count",
    color_continuous_scale="Reds",
    title="Number of APT Groups by Country (MITRE ATT&CK)",
    width=900,
    height=500,
)

fig.update_traces(
    textposition="outside",
    marker_line_color="black",
    marker_line_width=0.8,
    opacity=0.85
)

fig.update_layout(
    xaxis_title="Country",
    yaxis_title="APT Group Count",
    xaxis_tickangle=-45,
    title=dict(x=0.5, xanchor="center", font=dict(size=22)),
    plot_bgcolor="rgba(0,0,0,0)",
    paper_bgcolor="rgba(0,0,0,0)",
    margin=dict(l=60, r=20, t=60, b=120),
)
fig.show()